# 📊 Notebook 04 — Data Visualization
## Sales Data Analytics | Superstore Sales Dataset

---

### 🎯 Objective
Create professional, publication-quality visualizations using **Matplotlib** and **Seaborn**:

| # | Chart Type | Analysis |
|---|---|---|
| 1 | Bar Chart | Sales by Category |
| 2 | Bar Chart | Sub-Category Profit |
| 3 | Line Chart | Monthly Sales Trend |
| 4 | Line Chart | Yearly Growth |
| 5 | Pie Chart | Region Sales Share |
| 6 | Pie Chart | Segment Distribution |
| 7 | Heatmap | Sales by Month × Year |
| 8 | Scatter Plot | Sales vs Profit (Discount) |
| 9 | Box Plot | Sales Distribution by Category |
| 10 | Histogram | Profit Distribution |
| 11 | Correlation Matrix | Numeric Variables |
| 12 | Horizontal Bar | Top 10 Products |

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ── Style Config ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 130,
    'figure.facecolor' : '#0F1117',
    'axes.facecolor'   : '#1A1D27',
    'axes.edgecolor'   : '#2D3147',
    'axes.labelcolor'  : '#E0E0E0',
    'text.color'       : '#E0E0E0',
    'xtick.color'      : '#A0A0A0',
    'ytick.color'      : '#A0A0A0',
    'grid.color'       : '#2D3147',
    'grid.alpha'       : 0.4,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
})

COLORS  = ['#4FC3F7', '#AB47BC', '#66BB6A', '#FFA726', '#EF5350', '#26C6DA', '#FF7043']
ACCENT  = '#4FC3F7'
VIZ_DIR = Path('../visualizations')
VIZ_DIR.mkdir(exist_ok=True)

df = pd.read_csv('../dataset/superstore_clean.csv', parse_dates=['order_date', 'ship_date'])
print(f'✅ Dataset loaded: {df.shape[0]:,} rows')

## 4.1 — Sales & Profit by Category (Bar Chart)

In [ ]:
cat_data = (
    df.groupby('category')
    .agg(Total_Sales=('sales', 'sum'), Total_Profit=('profit', 'sum'))
    .reset_index()
    .sort_values('Total_Sales', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Sales & Profit by Category', fontsize=16, fontweight='bold', color='white', y=1.02)

# Sales bar
bars = axes[0].bar(cat_data['category'], cat_data['Total_Sales'] / 1000,
                   color=COLORS[:3], edgecolor='none', width=0.5)
axes[0].set_title('Total Revenue by Category', color='white', fontsize=13)
axes[0].set_ylabel('Revenue ($K)', color='white')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'${bar.get_height():,.0f}K', ha='center', va='bottom', fontsize=10, color='white')

# Profit bar with color based on sign
profit_colors = ['#66BB6A' if p > 0 else '#EF5350' for p in cat_data['Total_Profit']]
bars2 = axes[1].bar(cat_data['category'], cat_data['Total_Profit'] / 1000,
                    color=profit_colors, edgecolor='none', width=0.5)
axes[1].set_title('Total Profit by Category', color='white', fontsize=13)
axes[1].set_ylabel('Profit ($K)', color='white')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))
axes[1].axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
for bar in bars2:
    ypos = bar.get_height() + 1 if bar.get_height() >= 0 else bar.get_height() - 4
    axes[1].text(bar.get_x() + bar.get_width()/2, ypos,
                 f'${bar.get_height():,.0f}K', ha='center', va='bottom', fontsize=10, color='white')

plt.tight_layout()
plt.savefig(VIZ_DIR / '01_category_sales_profit.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 01_category_sales_profit.png')

## 4.2 — Sub-Category Profit (Horizontal Bar Chart)

In [ ]:
subcat_data = (
    df.groupby('sub_category')
    .agg(Total_Profit=('profit', 'sum'))
    .reset_index()
    .sort_values('Total_Profit')
)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#EF5350' if p < 0 else '#66BB6A' for p in subcat_data['Total_Profit']]
bars = ax.barh(subcat_data['sub_category'], subcat_data['Total_Profit'] / 1000,
               color=colors, edgecolor='none')

ax.set_title('Profit by Sub-Category\n(Red = Loss | Green = Profit)', fontsize=14, fontweight='bold', color='white')
ax.set_xlabel('Total Profit ($K)', color='white')
ax.axvline(0, color='white', linewidth=1, linestyle='--', alpha=0.6)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))

for bar, val in zip(bars, subcat_data['Total_Profit'] / 1000):
    xpos = val + 0.5 if val >= 0 else val - 0.5
    ha = 'left' if val >= 0 else 'right'
    ax.text(xpos, bar.get_y() + bar.get_height()/2, f'${val:,.1f}K',
            ha=ha, va='center', fontsize=9, color='white')

plt.tight_layout()
plt.savefig(VIZ_DIR / '02_subcategory_profit.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 02_subcategory_profit.png')

## 4.3 — Monthly Sales Trend (Line Chart)

In [ ]:
monthly = (
    df.groupby(['order_year', 'order_month'])
    .agg(Total_Sales=('sales', 'sum'), Total_Profit=('profit', 'sum'))
    .reset_index()
    .sort_values(['order_year', 'order_month'])
)
monthly['period'] = monthly['order_year'].astype(str) + '-' + monthly['order_month'].astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(range(len(monthly)), monthly['Total_Sales'] / 1000,
        color='#4FC3F7', linewidth=2.5, marker='o', markersize=4, label='Sales')
ax.fill_between(range(len(monthly)), monthly['Total_Sales'] / 1000,
                alpha=0.2, color='#4FC3F7')
ax.plot(range(len(monthly)), monthly['Total_Profit'] / 1000,
        color='#66BB6A', linewidth=2, linestyle='--', marker='s', markersize=3, label='Profit')

# Year separators
for yr in [2015, 2016, 2017]:
    idx = monthly[monthly['order_year'] == yr].index[0] - monthly.index[0]
    ax.axvline(idx, color='white', alpha=0.2, linewidth=1, linestyle=':')
    ax.text(idx, ax.get_ylim()[1] * 0.95, str(yr), color='white', fontsize=9, alpha=0.7)

ax.set_title('Monthly Sales & Profit Trend (2014–2017)', fontsize=14, fontweight='bold', color='white')
ax.set_ylabel('Amount ($K)', color='white')
ax.set_xlabel('Period', color='white')
ax.legend(framealpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))

# Sample x-tick labels
tick_idx = list(range(0, len(monthly), 3))
ax.set_xticks(tick_idx)
ax.set_xticklabels([monthly.iloc[i]['period'] for i in tick_idx], rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig(VIZ_DIR / '03_monthly_sales_trend.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 03_monthly_sales_trend.png')

## 4.4 — Region Sales Share (Pie Chart)

In [ ]:
region_data = df.groupby('region')['sales'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('Sales Distribution by Region & Segment', fontsize=15, fontweight='bold', color='white')

# Region pie
wedges, texts, autotexts = axes[0].pie(
    region_data.values,
    labels=region_data.index,
    autopct='%1.1f%%',
    colors=COLORS,
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor='#0F1117', linewidth=2)
)
for text in texts: text.set_color('white')
for at in autotexts: at.set_color('white'); at.set_fontsize(10)
axes[0].set_title('Sales Share by Region', color='white', fontsize=12, pad=15)

# Segment pie
seg_data = df.groupby('segment')['sales'].sum().sort_values(ascending=False)
wedges2, texts2, autotexts2 = axes[1].pie(
    seg_data.values,
    labels=seg_data.index,
    autopct='%1.1f%%',
    colors=['#4FC3F7', '#AB47BC', '#66BB6A'],
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor='#0F1117', linewidth=2)
)
for text in texts2: text.set_color('white')
for at in autotexts2: at.set_color('white'); at.set_fontsize(10)
axes[1].set_title('Sales Share by Customer Segment', color='white', fontsize=12, pad=15)

plt.tight_layout()
plt.savefig(VIZ_DIR / '04_region_segment_pie.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 04_region_segment_pie.png')

## 4.5 — Monthly Sales Heatmap (Year × Month)

In [ ]:
pivot = (
    df.groupby(['order_year', 'order_month'])['sales']
    .sum()
    .unstack('order_month')
    .fillna(0)
)
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    pivot / 1000,
    ax=ax,
    cmap='YlOrRd',
    annot=True,
    fmt='.0f',
    linewidths=0.5,
    linecolor='#0F1117',
    cbar_kws={'label': 'Sales ($K)', 'shrink': 0.8}
)
ax.set_title('Monthly Sales Heatmap — Year × Month ($K)', fontsize=14, fontweight='bold', color='white', pad=15)
ax.set_xlabel('Month', color='white')
ax.set_ylabel('Year', color='white')
ax.tick_params(colors='white')
plt.colorbar(ax.collections[0]).ax.yaxis.label.set_color('white')

plt.tight_layout()
plt.savefig(VIZ_DIR / '05_monthly_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 05_monthly_heatmap.png')

## 4.6 — Sales vs Profit Scatter Plot (colored by Discount)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

scatter = ax.scatter(
    df['sales'], df['profit'],
    c=df['discount'],
    cmap='RdYlGn_r',
    alpha=0.6,
    s=30,
    edgecolors='none'
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Discount Rate', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

ax.axhline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
ax.axvline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
ax.set_title('Sales vs Profit  (colored by Discount Rate)', fontsize=14, fontweight='bold', color='white')
ax.set_xlabel('Sales ($)', color='white')
ax.set_ylabel('Profit ($)', color='white')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Add annotation
ax.text(0.98, 0.02, '⚠️  High discount → negative profit',
        transform=ax.transAxes, color='#FFA726', fontsize=10, ha='right')

plt.tight_layout()
plt.savefig(VIZ_DIR / '06_sales_profit_scatter.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 06_sales_profit_scatter.png')

## 4.7 — Sales Distribution by Category (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

cat_order = df.groupby('category')['sales'].median().sort_values(ascending=False).index

box_data = [df[df['category'] == cat]['sales'].values for cat in cat_order]
bp = ax.boxplot(box_data, labels=cat_order, patch_artist=True, notch=True,
                medianprops=dict(color='white', linewidth=2))

for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_title('Sales Distribution by Category (Box Plot)', fontsize=14, fontweight='bold', color='white')
ax.set_xlabel('Category', color='white')
ax.set_ylabel('Sales ($)', color='white')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_yscale('log')
ax.text(0.01, 0.98, 'Log scale applied for readability',
        transform=ax.transAxes, color='#A0A0A0', fontsize=9, va='top')

plt.tight_layout()
plt.savefig(VIZ_DIR / '07_category_sales_boxplot.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 07_category_sales_boxplot.png')

## 4.8 — Profit Distribution (Histogram)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(df['profit'], bins=60, color='#4FC3F7', alpha=0.75, edgecolor='none')
ax.axvline(df['profit'].mean(),   color='#FFA726', linewidth=2, linestyle='--', label=f'Mean: ${df["profit"].mean():,.2f}')
ax.axvline(df['profit'].median(), color='#66BB6A', linewidth=2, linestyle='-.',  label=f'Median: ${df["profit"].median():,.2f}')
ax.axvline(0, color='#EF5350', linewidth=1.5, linestyle=':', label='Break Even')

ax.set_title('Profit Distribution — All Transactions', fontsize=14, fontweight='bold', color='white')
ax.set_xlabel('Profit ($)', color='white')
ax.set_ylabel('Frequency', color='white')
ax.legend(framealpha=0.3)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Stats annotation
loss_pct = (df['profit'] < 0).sum() / len(df) * 100
ax.text(0.98, 0.95, f'Loss transactions: {loss_pct:.1f}%',
        transform=ax.transAxes, color='#EF5350', fontsize=10, ha='right', va='top')

plt.tight_layout()
plt.savefig(VIZ_DIR / '08_profit_histogram.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 08_profit_histogram.png')

## 4.9 — Correlation Matrix

In [ ]:
num_cols = ['sales', 'quantity', 'discount', 'profit', 'shipping_days', 'profit_margin_pct', 'revenue_per_unit']
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    ax=ax,
    mask=mask,
    cmap='coolwarm',
    annot=True,
    fmt='.2f',
    vmin=-1, vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    linecolor='#0F1117',
    annot_kws={'size': 9},
)
ax.set_title('Correlation Matrix — Numeric Variables', fontsize=14, fontweight='bold', color='white', pad=15)
ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig(VIZ_DIR / '09_correlation_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 09_correlation_matrix.png')

## 4.10 — Top 10 Products by Revenue (Horizontal Bar)

In [ ]:
top_products = (
    df.groupby('product_name')
    .agg(Total_Sales=('sales', 'sum'), Total_Profit=('profit', 'sum'))
    .sort_values('Total_Sales', ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 7))
colors = ['#66BB6A' if p > 0 else '#EF5350' for p in top_products['Total_Profit']]

bars = ax.barh(
    top_products['product_name'].str[:45],
    top_products['Total_Sales'] / 1000,
    color=colors,
    edgecolor='none'
)

for bar, profit in zip(bars, top_products['Total_Profit'] / 1000):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.1f}K | P: ${profit:.1f}K',
            ha='left', va='center', fontsize=8.5, color='white')

ax.set_title('Top 10 Products by Revenue (Green = Profitable | Red = Loss)', fontsize=13, fontweight='bold', color='white')
ax.set_xlabel('Total Sales ($K)', color='white')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(VIZ_DIR / '10_top_products.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('📊 Chart saved: 10_top_products.png')

## 4.11 — Visualization Summary

All charts have been saved to the `visualizations/` folder:

| File | Chart |
|---|---|
| `01_category_sales_profit.png` | Sales & Profit by Category |
| `02_subcategory_profit.png` | Profit by Sub-Category |
| `03_monthly_sales_trend.png` | Monthly Trend Line |
| `04_region_segment_pie.png` | Region & Segment Pie Charts |
| `05_monthly_heatmap.png` | Year × Month Heatmap |
| `06_sales_profit_scatter.png` | Sales vs Profit Scatter |
| `07_category_sales_boxplot.png` | Sales Distribution Box Plot |
| `08_profit_histogram.png` | Profit Histogram |
| `09_correlation_matrix.png` | Correlation Matrix |
| `10_top_products.png` | Top 10 Products Bar Chart |

**➡️ Next Step**: Notebook 05 — Business Insights